In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🏥 Medical Triage Note Workflow

## What We're Building

A clinical note triage assistant that processes patient intake text:

1. **Parse intake** — extract symptoms, vitals, history from free text
2. **Structured summary** — organize into clinical format (SOAP-like)
3. **Missing-info questions** — identify gaps and generate clarifying questions
4. **Clinician handoff** — produce a triage note for the attending physician

## Architecture

```
Patient Intake Text
      │
      ▼
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│ Parse    │──▶│ Structure│──▶│ Missing  │──▶│ Clinician│
│ Intake   │   │ Summary  │   │ Info Qs  │   │ Handoff  │
└──────────┘   └──────────┘   └──────────┘   └──────────┘
```

## Key LangGraph Concept: State Accumulation

Each node enriches the state — the final handoff node has access to
everything produced by earlier nodes.

## Stack
- **LangGraph** — sequential pipeline with state enrichment
- **Ollama** — local LLM

> ⚠️ **Disclaimer**: This is an educational demo — NOT for real clinical use.

## Step 1 — Imports & Setup

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)
print("All imports successful!")

## Step 2 — State & Sample Patient Intakes

In [ ]:
class TriageState(TypedDict):
    intake_text: str           # Raw patient intake text
    parsed_data: str           # Extracted symptoms, vitals, history
    structured_summary: str    # SOAP-style organized note
    missing_info: str          # Questions for missing information
    triage_level: str          # emergent / urgent / semi-urgent / non-urgent
    handoff_note: str          # Final clinician-ready triage note


# Simulated patient intake records
patient_intakes = {
    "chest_pain": """
Patient: John M., 58-year-old male
Arrived at 14:32. Complaining of chest tightness that started
about 2 hours ago while climbing stairs. Says it feels like
"pressure", radiates to left arm. Slightly short of breath.
BP: 165/95, HR: 102, SpO2: 94%, Temp: 98.6F
History: hypertension (on lisinopril), type 2 diabetes.
Father had MI at age 55. Patient smokes 1/2 pack/day.
No known drug allergies. Last ate at noon. Aspirin given
by EMS en route.
""",

    "abdominal": """
Patient: Sarah K., 34-year-old female
Walk-in at 09:15. Reports lower right abdominal pain for
the past 18 hours, getting worse. Pain is sharp, rated 7/10.
Nausea but no vomiting. Low-grade fever. Lost appetite yesterday.
BP: 128/82, HR: 88, SpO2: 99%, Temp: 100.8F
No significant medical history. Takes birth control pills.
No known allergies. Last menstrual period 2 weeks ago.
""",

    "incomplete": """
Patient: Mike T., male, looks maybe 40s?
Came in complaining of headache. Says he's had it for
a few days. Took some Tylenol but it didn't help.
Seems a bit confused. No vitals taken yet.
Wife says he fell off a ladder last week but "seemed fine".
"""
}

print(f"Loaded {len(patient_intakes)} sample intakes")

## Step 3 — Define Nodes

### Node 1: Parse Intake
Extract structured data from free-text intake.

In [ ]:
parse_prompt = ChatPromptTemplate.from_template(
    """You are an ER triage nurse. Parse this patient intake into structured fields.

Intake Text:
{intake}

Extract:
- DEMOGRAPHICS: Name, age, sex, arrival time, mode of arrival
- CHIEF COMPLAINT: Primary reason for visit
- SYMPTOMS: List all reported symptoms with onset, duration, quality, severity
- VITALS: BP, HR, SpO2, Temp (note any that are abnormal)
- MEDICAL HISTORY: Past conditions, medications, surgical history
- FAMILY HISTORY: Relevant family medical history
- SOCIAL HISTORY: Smoking, alcohol, drugs
- ALLERGIES: Known allergies
- INTERVENTIONS: Any treatment already given
- MISSING: Note any standard fields that are NOT documented

Parsed data:"""
)


def parse_intake(state: TriageState) -> dict:
    print("📋 Node: parse_intake")
    chain = parse_prompt | llm | StrOutputParser()
    parsed = chain.invoke({"intake": state["intake_text"]})
    return {"parsed_data": parsed}

### Node 2: Structure into SOAP-like Summary

In [ ]:
structure_prompt = ChatPromptTemplate.from_template(
    """Organize these parsed patient details into a structured clinical summary.
Use this format:

## Subjective
(What the patient reports — symptoms, history, complaints)

## Objective
(Measurable data — vitals, observations)

## Assessment
(Preliminary assessment — most likely diagnosis, differentials to consider)
Include a TRIAGE LEVEL: emergent / urgent / semi-urgent / non-urgent

## Plan
(Recommended next steps — tests, imaging, consults, monitoring)

Parsed Data:
{parsed}

Structured summary:"""
)


def structure_summary(state: TriageState) -> dict:
    print("📊 Node: structure_summary")
    chain = structure_prompt | llm | StrOutputParser()
    summary = chain.invoke({"parsed": state["parsed_data"]})

    # Extract triage level
    triage_level = "urgent"  # default
    summary_lower = summary.lower()
    for level in ["emergent", "urgent", "semi-urgent", "non-urgent"]:
        if level in summary_lower:
            triage_level = level
            break

    print(f"     Triage level: {triage_level}")
    return {"structured_summary": summary, "triage_level": triage_level}

### Node 3: Identify Missing Information

In [ ]:
missing_prompt = ChatPromptTemplate.from_template(
    """Review this parsed patient intake and structured summary.
Identify any MISSING or INCOMPLETE information that a triage nurse
should collect before the physician sees the patient.

Original Intake:
{intake}

Parsed Data:
{parsed}

Generate specific questions the nurse should ask, grouped by priority:

CRITICAL (must ask before clinician sees patient):
IMPORTANT (should ask if time permits):
NICE-TO-HAVE (can be collected later):

Missing information questions:"""
)


def find_missing_info(state: TriageState) -> dict:
    print("❓ Node: find_missing_info")
    chain = missing_prompt | llm | StrOutputParser()
    missing = chain.invoke({
        "intake": state["intake_text"],
        "parsed": state["parsed_data"]
    })
    return {"missing_info": missing}

### Node 4: Generate Clinician Handoff Note

In [ ]:
handoff_prompt = ChatPromptTemplate.from_template(
    """Create a concise clinician handoff note for the attending physician.
This should be what the triage nurse verbally communicates during handoff.

Triage Level: {triage}

Structured Summary:
{summary}

Missing Information:
{missing}

Write a handoff note:
1. One-liner ("58M chest pain, triage: emergent")
2. Key findings (3-4 bullet points)
3. Flags/concerns (abnormal vitals, red flags)
4. Pending items (tests ordered, info still needed)

Clinician handoff note:"""
)


def clinician_handoff(state: TriageState) -> dict:
    print("🏥 Node: clinician_handoff")
    chain = handoff_prompt | llm | StrOutputParser()
    note = chain.invoke({
        "triage": state["triage_level"],
        "summary": state["structured_summary"],
        "missing": state["missing_info"]
    })
    return {"handoff_note": note}


print("All 4 nodes defined")

## Step 4 — Build & Compile the Graph

In [ ]:
workflow = StateGraph(TriageState)

workflow.add_node("parse_intake", parse_intake)
workflow.add_node("structure_summary", structure_summary)
workflow.add_node("find_missing_info", find_missing_info)
workflow.add_node("clinician_handoff", clinician_handoff)

workflow.set_entry_point("parse_intake")
workflow.add_edge("parse_intake", "structure_summary")
workflow.add_edge("structure_summary", "find_missing_info")
workflow.add_edge("find_missing_info", "clinician_handoff")
workflow.add_edge("clinician_handoff", END)

app = workflow.compile()
print("Medical triage workflow compiled")

## Step 5 — Process Patient Intakes

In [ ]:
def triage_patient(name: str, intake_text: str):
    print(f"\n{'='*60}")
    print(f"🏥 Triaging: {name}")
    print("="*60)

    result = app.invoke({
        "intake_text": intake_text,
        "parsed_data": "", "structured_summary": "",
        "missing_info": "", "triage_level": "", "handoff_note": "",
    })

    print(f"\n🚦 Triage Level: {result['triage_level'].upper()}")
    print(f"\n📋 Structured Summary:")
    print(result["structured_summary"])
    print(f"\n❓ Missing Info Questions:")
    print(result["missing_info"])
    print(f"\n🏥 Clinician Handoff Note:")
    print(result["handoff_note"])
    return result


# Case 1: Chest pain (likely emergent)
r1 = triage_patient("Chest Pain - John M.", patient_intakes["chest_pain"])

In [ ]:
# Case 2: Abdominal pain (likely urgent)
r2 = triage_patient("Abdominal Pain - Sarah K.", patient_intakes["abdominal"])

In [ ]:
# Case 3: Incomplete intake — showcases missing-info detection
r3 = triage_patient("Headache (Incomplete Intake) - Mike T.", patient_intakes["incomplete"])

In [ ]:
# Compare triage levels across all cases
print("\n" + "="*60)
print("📊 Triage Summary Dashboard")
print("="*60)
for label, result in [("Chest Pain", r1), ("Abdominal", r2), ("Incomplete", r3)]:
    level = result["triage_level"].upper()
    icon = {"EMERGENT": "🔴", "URGENT": "🟠", "SEMI-URGENT": "🟡", "NON-URGENT": "🟢"}.get(level, "⚪")
    print(f"  {icon} {label:25s} → {level}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Free-text parsing** | LLM extracts structured fields from messy intake notes |
| **SOAP structure** | Clinical standard: Subjective, Objective, Assessment, Plan |
| **Gap detection** | LLM identifies missing critical information |
| **State enrichment** | Each node adds to state; final node has all context |
| **Triage levels** | Emergent / Urgent / Semi-urgent / Non-urgent classification |

## 🔧 Extensions

- **HITL gate**: Interrupt before handoff for nurse to verify triage level
- **Allergy cross-check**: Compare prescribed interventions against known allergies
- **EMR integration**: Push structured note to electronic medical record
- **Priority routing**: Auto-page specific specialists based on assessment